# How to work with the database
The functions can be found in db.py. The database is built using [FAISS](https://github.com/facebookresearch/faiss) and [Redis](https://redis.io/)

In [1]:
import tqdm
from wp1.database.db import *
from wp1.models.model import *
from pathlib import Path
from PIL import Image
from wp1.dataset.dataset import open_and_convert_image

/home/axelle/Documents/Doctorat/Main research/WP1 Patch retrieval/code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/axelle/Documents/Doctorat/Main research/WP1 Patch retrieval/code/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


### Step 1: Device selection

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Step 2: Model creation
Place your pretrained weights in the weight_fold prior to starting the command, and replace the model name by the name of the model you plan in using - see *model_usage.ipynb* for more information

In [3]:
weight_path = Path("../../../../../weight_fold/resnet/model/last_epoch_new")
model = Model(model_name="resnet", weight=weight_path, device=device)

### Step 3: Building the database

In [4]:
# Creating a new database
db_name = "new_db" 
db = Database(db_name, model, load=False, device=device)

Creating a new FAISS index
Transferring index to GPU
Index transferred to GPU


In [5]:
load = True
if load:
    # Loading an existing database
    db_name = "new_db"
    db = Database(db_name, model, load=True, device=device)
    print(f"The database contains {db.__len__()} elements.")
else:
    # Add images
    dataset_path = Path("../../../../../Data/patches/CrC/NCT-CRC-HE-100K/")
    db.add_dataset(dataset_path)
    print(f"The database contains {db.__len__()} elements.")

Loaded 200001 keys into Redis.
Transferring index to GPU
Index transferred to GPU
The database contains 100000 elements.


### Step 4: Training the database (optional)

In [6]:
db.train_index()

Notice: The index is already trained (IVF). Skipping training.


### Step 5: Save the database

In [7]:
db.save()

Saving Redis data to new_db.json
Saved FAISS index and 200001 Redis keys to disk.


### Step 6: Search in the database

In [8]:
nb_neighbors = 10
query = Path("../../../../../Data/patches/CrC/CRC-VAL-HE-7K/ADI/ADI-TCGA-AAICEQFN.tif")
query_im = open_and_convert_image(query)
values, distances, t_model, t_search, t_transfer = db.search(query_im, nrt_neigh = nb_neighbors)

print("The ", nb_neighbors, " the closest to the query are: ", values,". They are at distances: ", distances)
print("The model took ", t_model, " to process the query, while the search in the database took ", t_search)

The  10  the closest to the query are:  ['../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-LPTCYGAM.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-KSRISHIV.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-DYWKCWME.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-QVTCHDPW.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-ATLWFGRP.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-FLSLCIRN.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-REMWFFFS.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-SFRNSHGS.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-PRWYVWSL.tif', '../../../../../Data/patches/CrC/NCT-CRC-HE-100K/ADI/ADI-ELWVCVFQ.tif'] . They are at distances:  [0.35274813 0.35325393 0.35609794 0.36426806 0.3718107  0.37601742
 0.3763423  0.3772725  0.37886956 0.3804704 ]
The model took  0.2181413173675537  to process the query, while the search in the database took  0.0